In [66]:
!pip install pypdf langchain langchain-community

In [67]:
!pip install langchain-community langchain-core

In [68]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the PDF
print("Loading the PDF...")
loader = PyPDFLoader("sample_document.pdf")
pages = loader.load()
print(f"Successfully loaded {len(pages)} pages.")

# 2. Configure the Text Splitter
# We break the text into 1,000-character chunks. 
# The 200-character overlap prevents sentences from being cut in half.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len
)

# 3. Process the Data
print("Chunking the text...")
chunks = text_splitter.split_documents(pages)

# 4. Verify the Output
print(f"Success! The document has been split into {len(chunks)} chunks.")
print("\n--- Preview of Chunk #1 ---")
print(chunks[0].page_content)

Loading the PDF...
Successfully loaded 15 pages.
Chunking the text...
Success! The document has been split into 52 chunks.

--- Preview of Chunk #1 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We pr

In [69]:
!pip install chromadb langchain-huggingface sentence-transformers

In [70]:
!pip install langchain-chroma
#this is the bridge between langchain and chromaDB

In [71]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

# 1. Initialize the Translator (Embedding Model)
# We are downloading a free, local AI model named "all-MiniLM-L6-v2".
# Its only job is to translate your English chunks into mathematical numbers.
print("Downloading the AI embedding model... (This may take a minute)")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Define the Database Location
# We are telling ChromaDB to save the database right next to your notebook
# inside a new folder named 'chroma_db'.
persist_directory = "./chroma_db"

# 3. Build and Save the Database
# We hand ChromaDB our text chunks and the AI model, and it builds the database.
print("Translating text to numbers and building the database...")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory
)

# 4. Verify Success
print(f"Success! Your Vector Database is built and saved locally at: {os.path.abspath(persist_directory)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Translating text to numbers and building the database...
Success! Your Vector Database is built and saved locally at: D:\Shubham_PythonProjects\RAG_search_engine_project\anaconda_projects\chroma_db


In [72]:
# Phase 3: The Retrieval System (Searching the Database)

# 1. Define the user's question
user_query = "What is the main topic of this document?"

# 2. Perform the Similarity Search
# We tell the database to search the vectors and return the top 3 closest matches (k=3)
print(f"Searching database for: '{user_query}'...\n")
search_results = vector_db.similarity_search(user_query, k=10)

# 3. Display the Top Result
print(f"Success! Found {len(search_results)} matching chunks. Here is the best match:\n")
print("-" * 60)
print(search_results[0].page_content)
print("-" * 60)

# 4. Show the Metadata (Data Engineering Bonus!)
# This proves to the user exactly where the AI got its answer
print(f"\nMetadata: {search_results[0].metadata}")

Searching database for: 'What is the main topic of this document?'...

Success! Found 10 matching chunks. Here is the best match:

------------------------------------------------------------
[25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated
corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993.
[26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In
Proceedings of the Human Language Technology Conference of the NAACL, Main Conference ,
pages 152–159. ACL, June 2006.
[27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention
model. In Empirical Methods in Natural Language Processing , 2016.
[28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive
summarization. arXiv preprint arXiv:1705.04304, 2017.
[29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Klein. Learning accurate, compact,
and i

In [73]:
!pip install langchain-groq

In [74]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# 1. Load the key from our hidden .env file(VScode)
load_dotenv()
my_api_key = os.getenv("GROQ_API_KEY")

# 2. Check if the key was found (prevents crashes)
if not my_api_key:
    print("❌ Error: GROQ_API_KEY not found in .env file!")
else:
    print("Connecting to Llama 3.1 via Groq...")
    
    # 3. Initialize the LLM using the secure variable
    llm = ChatGroq(
        api_key=my_api_key,
        temperature=0,
        model_name="llama-3.1-8b-instant"
    )
    
    print("✅ Connection successful! The AI is ready to read your documents.")

❌ Error: GROQ_API_KEY not found in .env file!


In [75]:
!pip install langchain-classic

In [76]:
# Updated for LangChain 1.x / 2026 standards
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
import os

# 1. Setup the Prompt
system_prompt = (
    "You are a professional AI assistant. "
    "Use ONLY the following pieces of retrieved context to answer the user's question. "
    "If the answer isn't in the context, say 'I don't know.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 2. Re-initialize the Retriever from your vector_db
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 3. Create the Chain using the classic library bridge
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# 4. Final Run!
question = "Summarize the key findings or the introduction of this document in 3 bullet points."
print(f"Asking the AI: '{question}'...\n")

response = rag_chain.invoke({"input": question})

print("-" * 60)
print("AI Assistant's Final Answer:")
print(response["answer"])
print("-" * 60)

Asking the AI: 'Summarize the key findings or the introduction of this document in 3 bullet points.'...

------------------------------------------------------------
AI Assistant's Final Answer:
I don't know. The provided context appears to be a list of references or citations, but it does not contain any text that can be summarized.
------------------------------------------------------------
